### Introduction to Machine Learning
Jesús Urtasun Elizari: MRC LMS 2026

**Chapter 1: What is learning. Exercise 5**

**Reinforcement learning**

An agent walks along a corridor of 5 positions and gets a reward only when it reaches the goal at the end.
Nobody tells it the right moves: it learns which action to take in each position from the rewards it collects.

**The maths**

The agent wants to collect as **much** reward as possible. Rewards that come later count a bit less, multiplied by a discount $\gamma < 1$:

$$G = r_1 + \gamma\, r_2 + \gamma^2\, r_3 + \dots$$

It keeps a table $Q(s, a)$ with how good it is to take action $a$ in state $s$. After every step, it moves the old value a little (learning rate $\alpha$) towards the reward received plus the best value of the next state $s'$:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt

# Check installation
print("All libraries imported successfully")
print(np.__version__)

In [ ]:
# Environment: a corridor with 5 positions (states)
#   0   1   2   3   4
#  [S] [ ] [ ] [ ] [G]
# The agent starts at 0, the goal is at 4
n_states = 5
goal = 4

# Two actions: 0 = left, 1 = right
actions = ['left', 'right']

def take_action(state, action):
    if action == 0:
        next_state = max(state - 1, 0)        # wall on the left
    else:
        next_state = min(state + 1, goal)
    reward = 1 if next_state == goal else 0   # reward only at the goal
    return next_state, reward

# Example: from state 3, go right
print(take_action(3, 1))

In [ ]:
# Q-table: expected long-term reward of each action in each state
# Rows are states, columns are actions. We start knowing nothing
Q = np.zeros((n_states, 2))
print("Q-table:\n", Q)

In [ ]:
# Parameters
alpha = 0.5    # learning rate: how much we update each time
gamma = 0.9    # discount: how much we care about future rewards
epsilon = 0.2  # exploration: probability of a random action

def choose_action(state):
    # Explore: random action (also when both actions look equally good)
    if np.random.rand() < epsilon or Q[state, 0] == Q[state, 1]:
        return np.random.randint(2)
    # Exploit: best action so far
    return np.argmax(Q[state])

In [ ]:
# Q-learning implementation
# Q(s, a) <- Q(s, a) + alpha * (reward + gamma * max Q(s_next) - Q(s, a))
print("Q-learning running")
np.random.seed(0)
steps_per_episode = []

for episode in range(30):
    state = 0
    steps = 0

    while state != goal:
        action = choose_action(state)
        next_state, reward = take_action(state, action)

        # Update the Q-table
        target = reward + gamma * np.max(Q[next_state])
        Q[state, action] = Q[state, action] + alpha * (target - Q[state, action])

        state = next_state
        steps += 1

    steps_per_episode.append(steps)
    print(f"Episode {episode}: {steps} steps to reach the goal")

In [ ]:
# Learned Q-table
print("Q-table after learning:\n", np.round(Q, 2))

# Policy: best action in each state
for state in range(goal):
    print(f"State {state}: go {actions[np.argmax(Q[state])]}")

In [ ]:
# Steps needed in each episode: fewer steps means the agent has learned
plt.figure(figsize = (8, 5))
plt.plot(steps_per_episode, color = 'purple', marker = 'o')
plt.axhline(4, color = 'gray', linestyle = 'dashed', label = 'Shortest path (4 steps)')
plt.xlabel('episode')
plt.ylabel('steps to reach the goal')
plt.title('Learning from rewards')
plt.legend()
plt.show()